In [22]:
print("h")

h


In [23]:
#Liberías

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

#ML para Arboles
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

#Modelos de ML
from sklearn.model_selection import train_test_split, cross_val_score
#StandarScaler lo que hace es escalar y transformar las features a media = 0 y desviacion = 1
from sklearn.preprocessing import StandardScaler, OrdinalEncoder

#Metricas de evaluación
from sklearn.metrics import (accuracy_score, classification_report, 
                             confusion_matrix, ConfusionMatrixDisplay)

In [24]:
df = pd.read_csv('student_habits_performance.csv')

df = df.drop(columns=['student_id'])
#Crear el target de clasificación
df['aprueba'] = (df['exam_score'] >= 60).astype(int)

df.head()

,age,gender,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score,aprueba
0,23,Female,0.0,1.2,1.1,No,85.0,8.0,Fair,6,Master,Average,8,Yes,56.2,0
1,20,Female,6.9,2.8,2.3,No,97.3,4.6,Good,6,High School,Average,8,No,100.0,1
2,21,Male,1.4,3.1,1.3,No,94.8,8.0,Poor,1,High School,Poor,1,No,34.3,0
3,23,Female,1.0,3.9,1.0,No,71.0,9.2,Poor,4,Master,Good,1,Yes,26.8,0
4,19,Female,5.0,4.4,0.5,No,90.9,4.9,Fair,3,Master,Good,1,No,66.4,1


In [25]:
#Tratamiento de nulos

moda_educacion = df['parental_education_level'].mode()[0]

df['parental_education_level'] = df['parental_education_level'].fillna(moda_educacion)

In [26]:
df.isnull().sum()

age                              0
gender                           0
study_hours_per_day              0
social_media_hours               0
netflix_hours                    0
part_time_job                    0
attendance_percentage            0
sleep_hours                      0
diet_quality                     0
exercise_frequency               0
parental_education_level         0
internet_quality                 0
mental_health_rating             0
extracurricular_participation    0
exam_score                       0
aprueba                          0
dtype: int64

In [27]:
# OrdinalEncoder para variables que tienen orden real



#Dieta
oe_diet = OrdinalEncoder(categories=[['Poor', 'Fair', 'Good']])
df['diet_quality'] = oe_diet.fit_transform(df[['diet_quality']]).reshape(-1)

#Internet
oe_net = OrdinalEncoder(categories=[['Poor', 'Average', 'Good']])
df['internet_quality'] = oe_net.fit_transform(df[['internet_quality']]).reshape(-1)


#Educación
oe_edu = OrdinalEncoder(categories=[['High School', 'Bachelor', 'Master']])
df['parental_education_level'] = oe_edu.fit_transform(df[['parental_education_level']]).reshape(-1)


#Binario de trabajo de medio tiempo
df['part_time_job'] = (df['part_time_job'] == 'Yes').astype(int)
df['extracurricular_participation'] = (df['extracurricular_participation'] == 'Yes').astype(int)


#One Hot Encoding para gender
df = pd.get_dummies(df, columns=['gender'], drop_first=True, dtype=int )


df.head(6)

,age,study_hours_per_day,social_media_hours,netflix_hours,part_time_job,attendance_percentage,sleep_hours,diet_quality,exercise_frequency,parental_education_level,internet_quality,mental_health_rating,extracurricular_participation,exam_score,aprueba,gender_Male,gender_Other
0,23,0.0,1.2,1.1,0,85.0,8.0,1.0,6,2.0,1.0,8,1,56.2,0,0,0
1,20,6.9,2.8,2.3,0,97.3,4.6,2.0,6,0.0,1.0,8,0,100.0,1,0,0
2,21,1.4,3.1,1.3,0,94.8,8.0,0.0,1,0.0,0.0,1,0,34.3,0,1,0
3,23,1.0,3.9,1.0,0,71.0,9.2,0.0,4,2.0,2.0,1,1,26.8,0,0,0
4,19,5.0,4.4,0.5,0,90.9,4.9,1.0,3,2.0,2.0,1,0,66.4,1,0,0
5,24,7.2,1.3,0.0,0,82.9,7.4,1.0,1,2.0,1.0,4,0,100.0,1,1,0


In [35]:
#División del dataset en entrenamineto y prueba
#Definir un alista de todas las features que el modelo usará para la regresión
#Excluimos a exam_score ya que ella es el target que queemos predecir

features = ['study_hours_per_day', 'social_media_hours', 'netflix_hours', 'part_time_job', 
                'attendance_percentage', 'sleep_hours','diet_quality', 'exercise_frequency', 
                'parental_education_level', 'internet_quality', 'mental_health_rating', 
                'extracurricular_participation','gender_Male', 'gender_Other']

X = df[features]

y = df['aprueba']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} estudiantes')
print(f'Prueba: {X_test.shape[0]} estudiantes')
print(f'Proporción aprueba en train: {y_train.mean():.1%}')
print(f'Proporción aprueba en test: {y_test.mean():.1%}')

Entrenamiento: 800 estudiantes
Prueba: 200 estudiantes
Proporción aprueba en train: 72.0%
Proporción aprueba en test: 72.0%


In [ ]:
#Constucción del árbol de Decisión CON RESTRICCIONES Y OTRO SIN ELLAS

